In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report

import data_utils
from base_utils_qwen import SequenceExtractor
from proto_utils_qwen import SingleHeadPrototypicalNetwork, MultiHeadPrototypicalNetwork, competition_scorer
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [2]:
# ============================================================
# CONFIGURATION - CHANGE THIS TO SWITCH MODES
# ============================================================

# Set this to "single" or "multi"
mode = "multi"  # "single" for binary+gesture, "multi" for multi-head

single_target = "bfrb"  # For single mode: "gesture", "gesture_action", etc.

# For MULTI mode: which heads to predict
multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb"  # For multi mode: what to evaluate F1 on

pipe_name = "extractor"
proto_name = "model"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 5
n_jobs = 1
train_size = 0.25
error_score_constant = np.nan
verbose = 4

cv = GroupKFold(n_splits=n_splits)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Mode: {mode}")
print(f"Search mode: {search_mode}")

Mode: multi
Search mode: grid


In [3]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data


In [4]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
bad_q = norm.squeeze() == 0
q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
q_wxyz = q_wxyz / norm

q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
euler_xyz[:, [1, 2]] *= -1.0
q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed

print(f"left-handed corrected: {left_handed_mask.sum()} rows")

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
print(f"upside-down corrected: {upside_down_mask.sum()} rows")

train_df = train_df.drop(columns=["handedness"])

left-handed corrected: 71352 rows
upside-down corrected: 12257 rows


In [5]:
# Create target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, train_pct=train_size, test_pct=(1 - train_size), random_state=random_state
)

print(f"Train sequences: {train_sample_df['sequence_id'].nunique()}")
print(f"Hold Out sequences: {hold_out_df['sequence_id'].nunique()}")

Train: 1458 seqs | 17.9%
Test:  4333 seqs  | 53.2%
Train sequences: 1458
Hold Out sequences: 4333


In [6]:
train_sample_df.groupby(['bfrb', 'orientation']).agg(
    {'sequence_id':'nunique'})

sequence_id
bfrb                     orientation                                 
Above ear - pull hair    Lie on Back                               20
                         Lie on Side - Non Dominant                22
                         Seated Lean Non Dom - FACE DOWN           21
                         Seated Straight                           18
Cheek - pinch skin       Lie on Back                               19
                         Lie on Side - Non Dominant                17
                         Seated Lean Non Dom - FACE DOWN           21
                         Seated Straight                           24
Eyebrow - pull hair      Lie on Back                               13
                         Lie on Side - Non Dominant                18
                         Seated Lean Non Dom - FACE DOWN           25
                         Seated Straight                           25
Eyelash - pull hair      Lie on Back                               20
                         Lie on Side - Non Dominant                23
                         Seated Lean Non Dom - FACE DOWN           21
                         Seated Straight                           17
Forehead - pull hairline Lie on Back                               18
                         Lie on Side - Non Dominant                22
                         Seated Lean Non Dom - FACE DOWN           25
                         Seated Straight                           16
Forehead - scratch       Lie on Back                               19
                         Lie on Side - Non Dominant                26
                         Seated Lean Non Dom - FACE DOWN           23
                         Seated Straight                           13
Neck - pinch skin        Lie on Back                               18
                         Lie on Side - Non Dominant                16
                         Seated Lean Non Dom - FACE DOWN           15
                         Seated Straight                           32
Neck - scratch           Lie on Back                               22
                         Lie on Side - Non Dominant                27
                         Seated Lean Non Dom - FACE DOWN           13
                         Seated Straight                           19
non_bfrb                 Lie on Back                               87
                         Lie on Side - Non Dominant               108
                         Seated Lean Non Dom - FACE DOWN          264
                         Seated Straight                          351

In [7]:
sequence_extractor = SequenceExtractor(
    acc_modes='raw'
)

if mode == "single":
    # BinaryPlusGesturePrototypicalNetwork uses is_target and gesture automatically
    model = SingleHeadPrototypicalNetwork(
        target=single_target,
    )
else:
    model = MultiHeadPrototypicalNetwork(
        primary_target=primary_target,
    )

pipeline = Pipeline([
    ("extractor", sequence_extractor),
    ("model", model),
])

In [ ]:
if search_mode == "bayesian":
    # ============================================================
    # 1. EXTRACTOR PARAMETERS (AdvancedMultiDomainSequenceExtractor)
    # ============================================================
    base_extractor_params = {
        # Multi-Domain Feature Combinations (Honeycomb Structure)
        f"{pipe_name}__acc_modes": Categorical([
            "raw|velocity|displacement|jerk", 
            "smoothed|velocity|jerk", 
            "raw|smoothed|displacement"
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|rot6d|angular_velocity", 
            "quaternion|delta_euler", 
            "quaternion|euler|angular_velocity"
        ]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff", "diff", "centered"]),
        
        # Advanced Motion Filtering (Kalman & Dead Reckoning)
        f"{pipe_name}__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        
        # Signal Processing & Sequence Padding
        f"{pipe_name}__sampling_rate": Categorical([20, 100]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([None, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__window_size": Integer(5, 21),
        f"{pipe_name}__smooth_alpha": Real(0.1, 0.9),
        f"{pipe_name}__maxlen": Categorical([120, 160, 200]),
        f"{pipe_name}__padding_value": Categorical([-999.0]),
    }
    
    # ============================================================
    # 2. MODEL PARAMETERS (Proto Utils: Single/Multi Head)
    # ============================================================
    base_model_params = {
        # Backbone Architecture
        f"{proto_name}__backbone_type": Categorical(["1dcnn", "attention", "lstm"]),
        f"{proto_name}__filters": Categorical(["64-128", "128-256", "64-128-256"]),
        f"{proto_name}__kernels": Categorical(["3-3", "5-3", "7-5-3"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2"]),
        
        # Regularization & Embedding
        f"{proto_name}__dropout": Real(0.1, 0.5),
        f"{proto_name}__embed_dim": Categorical([64, 128, 256]),
        f"{proto_name}__lstm_units": Integer(64, 256), # Used if backbone is lstm/bidirectional
        
        # Optimization & Episode Sampling
        f"{proto_name}__learning_rate": Real(1e-5, 1e-2, prior="log-uniform"),
        f"{proto_name}__batch_size": Categorical([16, 32]),
        f"{proto_name}__epochs": Categorical([50, 100]),
        f"{proto_name}__patience": Categorical([10, 15]),
        
        f"{proto_name}__n_way": Categorical([9, 18]), 
        f"{proto_name}__n_support": Categorical([5, 10, 20, 40]),
        f"{proto_name}__n_query": Categorical([10, 15, 20, 40]),
        
        # Temporal Augmentations (New in proto_utils)
        f"{proto_name}__use_mixup": Categorical([True, False]),
        f"{proto_name}__mixup_alpha": Real(0.1, 0.6),
        f"{proto_name}__use_time_mask": Categorical([True, False]),
        f"{proto_name}__time_mask_ratio": Real(0.05, 0.2),
        f"{proto_name}__use_gaussian_noise": Categorical([True, False]),
        f"{proto_name}__noise_std": Real(0.001, 0.05, prior="log-uniform"),
    }
    
    if mode == "single":
        param_space = {
            **base_extractor_params,
            **base_model_params,
            f"{proto_name}__target": Categorical([single_target]),
        }
    else:  
        heads_tuple = tuple(multi_heads)
        param_space = {
            **base_extractor_params,
            **base_model_params,
            f"{proto_name}__sub_heads": Categorical([heads_tuple]),
            f"{proto_name}__primary_target": Categorical([primary_target]),
        }

else: # Grid Search
    param_space = {
        # ============================================================
        # 1. EXTRACTOR PARAMETERS (AdvancedMultiDomainSequenceExtractor)
        # ============================================================
        f"{pipe_name}__acc_modes": ["smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|rot6d|angular_velocity"],
        f"{pipe_name}__tof_modes": ["sensor_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        
        f"{pipe_name}__motion_filter_mode": ["extended_kalman"],
        f"{pipe_name}__kalman_process_noise": [1e-3],
        f"{pipe_name}__kalman_measurement_noise": [1e-1],
        f"{pipe_name}__use_dead_reckoning": [True],
        f"{pipe_name}__dead_reckoning_detrend": [True],
        
        f"{pipe_name}__sampling_rate": [100],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [20],
        f"{pipe_name}__smooth_alpha": [0.5],
        f"{pipe_name}__maxlen": [160],
        f"{pipe_name}__padding_value": [-999.0],
        
        # ============================================================
        # 2. MODEL PARAMETERS (Proto Utils)
        # ============================================================
        f"{proto_name}__backbone_type": ["2dcnn"],
        f"{proto_name}__filters": ["128"],
        f"{proto_name}__kernels": ["3"],
        f"{proto_name}__pools": ["2"],
        
        f"{proto_name}__dropout": [0.0],
        f"{proto_name}__embed_dim": [64],
        f"{proto_name}__lstm_units": [128],
        
        f"{proto_name}__learning_rate": [5e-3],
        f"{proto_name}__batch_size": [16],
        f"{proto_name}__epochs": [100],
        f"{proto_name}__patience": [15],
        
        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [20],
        f"{proto_name}__n_query": [20],
        
        # Augmentations (Fixed for grid)
        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__mixup_alpha": [0.4],
        f"{proto_name}__use_time_mask": [False],
        f"{proto_name}__time_mask_ratio": [0.1],
        f"{proto_name}__use_gaussian_noise": [True],
        f"{proto_name}__noise_std": [0.01],

        # ============================================================
        # 3. MODE-SPECIFIC TARGETS
        # ============================================================
        f"{proto_name}__target": [single_target] if mode == "single" else None,
        f"{proto_name}__primary_target": [primary_target] if mode != "single" else None,
    }

    # Only add target parameters when in correct mode
    if mode == "single":
        param_space[f"{proto_name}__target"] = [single_target]
    else:
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]

In [9]:
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    # For multi mode, include all heads plus the primary target for lookup
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols:
        cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", primary_target]].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
groups = X_train["sequence_id"]

print(f"MODE: {mode}")
print(f"Single target: {single_target if mode=='single' else 'N/A'}")
print(f"Multi heads: {multi_heads if mode=='multi' else 'N/A'}")
print(f"Primary target: {primary_target if mode=='multi' else 'N/A'}")

print(f"y_train columns: {y_train.columns.tolist()}")

MODE: multi
Single target: N/A
Multi heads: ['gesture_action', 'orientation', 'gesture_position']
Primary target: bfrb
y_train columns: ['sequence_id', 'gesture_action', 'orientation', 'gesture_position', 'bfrb']


In [10]:
if search_mode == "bayesian":
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=competition_scorer,
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=competition_scorer,
        cv=cv,
        n_jobs=n_jobs,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

Running grid search...


TypeError: Parameter grid for parameter 'model__target' needs to be a list or a numpy array, but got None (of type NoneType) instead. Single values need to be wrapped in a list with one element.

In [ ]:
# ============================================================
# SAVE CV RESULTS TO CSV
# ============================================================

# Convert search results to DataFrame
cv_results_df = pd.DataFrame(search.cv_results_)

# Add metadata columns
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp

# Define save path
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"

# Save to CSV
cv_results_df.to_csv(results_path, index=False)

print(f"CV results saved to: {results_path}")
print(f"Shape: {cv_results_df.shape}")
print(f"Columns: {cv_results_df.columns.tolist()}")

In [ ]:
# ============================================================
# EVALUATE ON HOLDOUT TEST SET - WORKS FOR BOTH MODES
# ============================================================

# Prepare test labels - USE BFRB COLUMN (it has 'non_bfrb' + gestures)
y_test_true = hold_out_df[["sequence_id", "is_target", "bfrb"]].copy()

# Predict (Returns 1 prediction per sequence -> length 4333)
y_pred = best_model.predict(X_test)

# IMPORTANT: Aggregate y_test_true to the sequence level to match y_pred!
# SequenceExtractor groups by sequence_id and sorts them alphabetically/numerically.
# We must do the exact same thing to our ground truth labels.
y_test_true_seq = (
    y_test_true
    .drop_duplicates(subset=["sequence_id"])
    .sort_values("sequence_id")
    .reset_index(drop=True)
)

# Binary F1 (target vs non-target)
y_true_binary = y_test_true_seq["is_target"].values.astype(int)
y_pred_binary = (y_pred != "non_bfrb").astype(int)
binary_f1 = f1_score(y_true_binary, y_pred_binary)

# Gesture Macro F1 (only target/BFRB sequences)
target_mask = y_true_binary == 1
if target_mask.sum() > 0:
    gesture_f1 = f1_score(
        y_test_true_seq.loc[target_mask, "bfrb"].values, 
        y_pred[target_mask], 
        average="macro"
    )
else:
    gesture_f1 = 0.0

competition_score = (binary_f1 + gesture_f1) / 2

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)
print(f"Binary F1 (non_bfrb vs bfrb): {binary_f1:.4f}")
print(f"BFRB Gesture Macro F1: {gesture_f1:.4f}")
print(f"COMPETITION SCORE: {competition_score:.4f}")

if target_mask.sum() > 0:
    print("\n" + "-"*40)
    print("BFRB Gesture Classification Report")
    print("-"*40)
    print(classification_report(
        y_test_true_seq.loc[target_mask, "bfrb"].values, 
        y_pred[target_mask]
    ))

# Save results (Now using sequence-level arrays so lengths match perfectly)
holdout_results = pd.DataFrame({
    "sequence_id": y_test_true_seq["sequence_id"].values,
    "is_target_true": y_true_binary,
    "is_target_pred": y_pred_binary,
    "bfrb_true": y_test_true_seq["bfrb"].values,
    "bfrb_pred": y_pred,
})
holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_results.to_csv(holdout_results_path, index=False)
print(f"\nHoldout predictions saved to: {holdout_results_path}")

In [ ]:
unique_labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=unique_labels, 
            yticklabels=unique_labels)
plt.title(f"Confusion Matrix - {mode} mode\nTarget: {single_target if mode=='single' else primary_target}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = results_dir / f"confusion_matrix_{mode}_{timestamp}.png"
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved to: {cm_path}")

print("\n" + "="*60)
print("EXPERIMENT COMPLETE")
print("="*60)